In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/08_integration/01_copilot_orchestrator.py

In [0]:
# ================================================================
# PHASE 20 — COPILOT PIPELINE
# ================================================================

print("=" * 70)
print("PHASE 20 — COPILOT PIPELINE")
print("=" * 70)

print()
print("Purpose:")
print("SQL + RAG + Hybrid orchestration pipeline")

In [0]:
# ================================================================
# COPILOT DEPENDENCY CHECK
# ================================================================

print("=" * 70)
print("COPILOT DEPENDENCY CHECK")
print("=" * 70)

required_functions = [
    "classify_question",
    "generate_sql_request",
    "validate_sql",
    "execute_sql",
    "generate_question_embedding",
    "retrieve_documents",
    "build_rag_context",
    "generate_rag_answer",
    "decompose_hybrid_question",
    "run_sql_route",
    "run_hybrid_route",
    "assemble_hybrid_answer",
    "ask_copilot"
]

failed_dependencies = []

for function_name in required_functions:

    available = callable(
        globals().get(function_name)
    )

    print(
        f"{'PASS' if available else 'FAIL'} - "
        f"{function_name}"
    )

    if not available:
        failed_dependencies.append(
            function_name
        )

print()
print("Total dependencies:", len(required_functions))
print("Failed dependencies:", len(failed_dependencies))

if failed_dependencies:

    raise RuntimeError(
        "Copilot pipeline cannot continue. "
        "Missing functions: "
        + ", ".join(failed_dependencies)
    )

print()
print("Copilot dependency check: PASS")

In [0]:
# ================================================================
# PIPELINE CONFIGURATION
# ================================================================

import json
import time

PIPELINE_NAME = "genai_data_analyst_copilot"

PIPELINE_VERSION = "1.0"

TEST_QUESTIONS = {
    "sql": "Which region generated the highest revenue?",
    "rag": "What is the discount policy?",
    "hybrid": (
        "Which region generated the highest revenue "
        "and what discount policy applies there?"
    )
}

print("=" * 70)
print("PIPELINE CONFIGURATION")
print("=" * 70)

print("Pipeline:", PIPELINE_NAME)
print("Version:", PIPELINE_VERSION)

print()
print("Configured tests:")

for route, question in TEST_QUESTIONS.items():

    print(
        f"{route.upper():10} -> {question}"
    )

In [0]:
# ================================================================
# COPILOT PIPELINE RUNNER
# ================================================================

def run_copilot_pipeline(question):
    """
    Execute the complete copilot pipeline.

    Routes:
        SQL
        RAG
        Hybrid
        Unsupported
    """

    start_time = time.time()

    result = {
        "success": False,
        "pipeline": PIPELINE_NAME,
        "pipeline_version": PIPELINE_VERSION,
        "question": question,
        "route": None,
        "answer": None,
        "sql": None,
        "data": None,
        "sources": [],
        "error": None,
        "execution_time_ms": None
    }

    try:

        if not question or not str(question).strip():

            result["error"] = "Question is empty."

            return result

        # --------------------------------------------------------
        # Execute copilot
        # --------------------------------------------------------

        copilot_result = ask_copilot(
            question
        )

        if copilot_result is None:

            result["error"] = (
                "ask_copilot() returned None."
            )

            return result

        if not isinstance(copilot_result, dict):

            result["error"] = (
                "ask_copilot() returned "
                f"{type(copilot_result).__name__} "
                "instead of dictionary."
            )

            return result

        # --------------------------------------------------------
        # Copy response
        # --------------------------------------------------------

        result["success"] = copilot_result.get(
            "success",
            False
        )

        result["route"] = copilot_result.get(
            "route"
        )

        result["answer"] = copilot_result.get(
            "answer"
        )

        result["sql"] = copilot_result.get(
            "sql"
        )

        result["data"] = copilot_result.get(
            "data"
        )

        result["sources"] = copilot_result.get(
            "sources",
            []
        )

        result["error"] = copilot_result.get(
            "error"
        )

        return result

    except Exception as e:

        result["error"] = (
            f"{type(e).__name__}: {str(e)}"
        )

        return result

    finally:

        result["execution_time_ms"] = round(
            (time.time() - start_time) * 1000,
            2
        )

In [0]:
# ================================================================
# TEST 1 — SQL PIPELINE
# ================================================================

print("=" * 70)
print("TEST 1 — SQL PIPELINE")
print("=" * 70)

sql_result = run_copilot_pipeline(
    TEST_QUESTIONS["sql"]
)

print(
    json.dumps(
        {
            "success": sql_result["success"],
            "question": sql_result["question"],
            "route": sql_result["route"],
            "answer": sql_result["answer"],
            "sql": sql_result["sql"],
            "error": sql_result["error"],
            "execution_time_ms": sql_result["execution_time_ms"]
        },
        indent=2,
        default=str
    )
)

sql_passed = (
    sql_result["success"]
    and sql_result["route"] == "sql"
    and sql_result["error"] is None
)

print()
print(
    "SQL pipeline:",
    "PASS" if sql_passed else "FAIL"
)

In [0]:
# ================================================================
# TEST 2 — RAG PIPELINE
# ================================================================

print("=" * 70)
print("TEST 2 — RAG PIPELINE")
print("=" * 70)

rag_result = run_copilot_pipeline(
    TEST_QUESTIONS["rag"]
)

print(
    json.dumps(
        {
            "success": rag_result["success"],
            "question": rag_result["question"],
            "route": rag_result["route"],
            "answer": rag_result["answer"],
            "sources": rag_result["sources"],
            "error": rag_result["error"],
            "execution_time_ms": rag_result["execution_time_ms"]
        },
        indent=2,
        default=str
    )
)

rag_passed = (
    rag_result["success"]
    and rag_result["route"] == "rag"
    and rag_result["answer"] is not None
    and rag_result["error"] is None
)

print()
print(
    "RAG pipeline:",
    "PASS" if rag_passed else "FAIL"
)

In [0]:
# ================================================================
# TEST 3 — HYBRID PIPELINE
# ================================================================

print("=" * 70)
print("TEST 3 — HYBRID PIPELINE")
print("=" * 70)

hybrid_result = run_copilot_pipeline(
    TEST_QUESTIONS["hybrid"]
)

print(
    json.dumps(
        {
            "success": hybrid_result["success"],
            "question": hybrid_result["question"],
            "route": hybrid_result["route"],
            "answer": hybrid_result["answer"],
            "sql": hybrid_result["sql"],
            "error": hybrid_result["error"],
            "execution_time_ms": hybrid_result["execution_time_ms"]
        },
        indent=2,
        default=str
    )
)

hybrid_passed = (
    hybrid_result["success"]
    and hybrid_result["route"] == "hybrid"
    and hybrid_result["answer"] is not None
    and hybrid_result["error"] is None
)

print()
print(
    "Hybrid pipeline:",
    "PASS" if hybrid_passed else "FAIL"
)

In [0]:
# ================================================================
# PHASE 20 — PIPELINE VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 20 — COPILOT PIPELINE VALIDATION")
print("=" * 70)

pipeline_tests = [
    ("SQL", sql_result, sql_passed),
    ("RAG", rag_result, rag_passed),
    ("HYBRID", hybrid_result, hybrid_passed)
]

successful_tests = 0

for route_name, result, passed in pipeline_tests:

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{route_name}"
    )

    if passed:
        successful_tests += 1

print()
print(
    "Total pipeline tests:",
    len(pipeline_tests)
)

print(
    "Successful pipeline tests:",
    successful_tests
)

print(
    "Failed pipeline tests:",
    len(pipeline_tests) - successful_tests
)

print()

if successful_tests == len(pipeline_tests):

    print(
        "PHASE 20 STATUS: PASS ✓"
    )

else:

    print(
        "PHASE 20 STATUS: FAIL ✗"
    )

In [0]:
# ================================================================
# PIPELINE SUMMARY
# ================================================================

print("=" * 70)
print("COPILOT PIPELINE SUMMARY")
print("=" * 70)

summary = {
    "pipeline": PIPELINE_NAME,
    "version": PIPELINE_VERSION,
    "sql": {
        "success": sql_result["success"],
        "route": sql_result["route"]
    },
    "rag": {
        "success": rag_result["success"],
        "route": rag_result["route"]
    },
    "hybrid": {
        "success": hybrid_result["success"],
        "route": hybrid_result["route"]
    }
}

print(
    json.dumps(
        summary,
        indent=2,
        default=str
    )
)